# Week 6 - Neural Network I - Homework

This homework builds on the material in the notes on [neural networks](https://openlearninglibrary.mit.edu/courses/course-v1:MITx+6.036+1T2019/courseware/Week6/neural_networks/1) up through and including section 6 on loss functions.

In particular, in this homework we consider neural networks with multiple layers. Each layer has multiple inputs and outputs, and can be broken down into two parts:

- A **linear module** that implements a linear transformation:

  $$
  z_j = \left(\sum_{i=1}^{m} x_i w_{i,j}\right) + w_{0,j}
  $$

  specified by a weight matrix $W$ and a bias vector $W_0$. The output is

  $$
  [z_1,\ldots,z_n]^T.
  $$

- An **activation module** that applies an activation function to the outputs of the linear module for some activation function $f$, such as Tanh or ReLU in the hidden layers or Softmax (see below) at the output layer. We write the output as:

  $$
  [f(z_1),\ldots,f(z_n)]^T
  $$

  although technically, for some activation functions such as softmax, each output will depend on all the $z_i$, not just one.

We will use the following notation for quantities in a network:

- Inputs to the network are $x_1,\ldots,x_d$.
- Number of layers is $L$.
- There are $m^l$ inputs to layer $l$.
- There are $n^l = m^{l+1}$ outputs from layer $l$.
- The weight matrix for layer $l$ is $W^l$, an $m^l \times n^l$ matrix, and the bias vector (offset) is $W_0^l$, an $n^l \times 1$ vector.
- The outputs of the linear module for layer $l$ are known as **pre-activation values** and denoted $z^l$.
- The activation function at layer $l$ is $f^l(\cdot)$.
- Layer $l$ activations are

  $$
  a^l = [f^l(z_1^l),\ldots,f^l(z_{n^l}^l)]^T.
  $$

- The output of the network is the values

  $$
  a^L = [f^L(z_1^L),\ldots,f^L(z_{n^L}^L)]^T.
  $$

- Loss function $Loss(a,y)$ measures the loss of output values $a$ when the target is $y$.

## 1) Loss functions and output activations: classification

### 1.1) Hinge loss, linear activation

**1.1.A)** Write a short program to compute the gradient of the loss function with respect to the weight vector (not the bias): $\nabla_w L Loss(a_1^L,y)$ when $Loss(a,y)=L_h(ya)$.

In [1]:
import numpy as np

In [ ]:
def hinge_loss_grad(x, y, a):
  if y * a > 1:
    return np.zeros_like(x)
  
  return - y * a 

<div align="center">
    <img src="../assets/homeworks/1_1_1_A.png" alt="1_1_1_A" style="border-radius: 10px" />
</div>

### 1.2) Log loss, sigmoidal activation

1.2.A) What is an expression for the derivative of the sigmoid with respect to $z$, expressed as a function of $z$, its input?

$$
\frac{e^{-z}}{1+e^{-z}}
$$

<div align="center">
    <img src="../assets/homeworks/2_1_2_A.png" alt="2_1_2_A" style="border-radius: 10px" />
</div>

1.2.B) What is an expression for the derivative of the sigmoid with respect to $z$, but this time expressed as a function of $o=\sigma(z)$, its output?

Hint: Think about the expression

$$
1-\frac{1}{1+e^{-z}}
$$

$$
o - o^{2}
$$

<div align="center">
    <img src="../assets/homeworks/3_1_2_B.png" alt="3_1_2_B" style="border-radius: 10px" />
</div>

**In this model, we will consider positive points to have label +1, and negative points to have label 0**.

We need a loss function that works well when we are predicting probabilities. A good choice is to ask what probability is assigned to the correct label.

We will interpret the value outputted by our classifier as the probability that the example is positive. So, if the output value is $a$ and the true label is $+1$, then the probability assigned to the true label is $a$; on the other hand, if the true label is $0$, then the probability assigned to the true label is $1-a$.

Because we actually will be interested in the probability of the predictions on the whole data set, we'd want to choose weights to **maximize**

$$
\prod_t P(a^{(t)},y^{(t)})
$$

where $P(a^{(t)},y^{(t)})$ is the probability that the network predicts the correct label for data point $(t)$.

Using a notational trick (which turns an *if* expression into a product) that might seem unmotivated now, but will be useful later, we can write the probability $P(a,y)$ as

$$
P(a,y)=a^y(1-a)^{1-y}.
$$

<div align="center">
    <img src="../assets/homeworks/4_1_2_C.png" alt="4_1_2_C" style="border-radius: 10px" />
</div>

<div align="center">
    <img src="../assets/homeworks/5_1_2_D.png" alt="5_1_2_D" style="border-radius: 10px" />
</div>

<div align="center">
    <img src="../assets/homeworks/6_1_2_E.png" alt="6_1_2_E" style="border-radius: 10px" />
</div>

In fact, because $\log$ is a monotonic function, the same weights that maximize the product of the probabilities will minimize the *negative log likelihood* ("likelihood" is the same as probability; we just use that name here because the phrase is an idiom in machine learning, abbreviated NLL):

$$
Loss(a,y)=NLL(a,y)=-y\log a-(1-y)\log(1-a).
$$

Our objective function (over our $n$ data points) will then be

$$
\sum_t NLL(a^{(t)},y^{(t)})
=
-\sum_{t=1}^{n}
\left[
y^{(t)}\log a^{(t)}
+
(1-y^{(t)})\log(1-a^{(t)})
\right].
$$

Remember that $a^{(t)}$ is our model's output for training example $t$, and $y^{(t)}$ is the true label (+1 or 0).

Now, we can think about a single unit with a sigmoidal activation function, trained to minimize NLL. So,

$$
a_1^L
=
\sigma\left(
\sum_k w_{k,1}^L x_k + w_{0,1}^L
\right).
$$

In this case, we have $L=1$.

1.2.F) Write a formula for the gradient of the NLL with respect to the first weight, $\nabla_{w_{1,1}^L}NLL(a_1^L,y)$, for a single training example.

Hint: consider using the chain rule; the final answer (expression) is very short.

$$
\nabla_{w_{1,1}^L}NLL(a_1^L,y)
=
(a_1^L-y)x_1
$$

<div align="center">
    <img src="../assets/homeworks/7_1_2_F.png" alt="7_1_2_F" style="border-radius: 10px" />
</div>

1.2.G) Write a formula for the gradient of the NLL with respect to the full weight vector, $\nabla_{W^L}NLL(a_1^L,y)$, for a single training example.

$$
\nabla_{W^L}NLL(a_1^L,y)
=
(a_1^L-y)x
$$

<div align="center">
    <img src="../assets/homeworks/8_1_2_G.png" alt="8_1_2_G" style="border-radius: 10px" />
</div>